In [ ]:
# Install required packages
!pip install --upgrade --quiet gradio ipywidgets pymupdf rapidocr opencv-python-headless onnxruntime torch transformers accelerate sentencepiece

print('✓ Packages installed!')

# Gradio demo: OCR (fancy)

In [1]:
from functools import lru_cache
from pathlib import Path
from tempfile import NamedTemporaryFile, TemporaryDirectory

import fitz
import gradio as gr
from rapidocr import RapidOCR


DPI = 200
rapidocr = RapidOCR()


def render_pages(pdf_path, pages, image_dir):
    zoom = DPI / 72
    matrix = fitz.Matrix(zoom, zoom)
    image_paths = []

    with fitz.open(pdf_path) as pdf:
        for page_number in pages:
            page = pdf.load_page(page_number - 1)
            pixmap = page.get_pixmap(matrix=matrix, alpha=False)
            image_path = image_dir / f"page-{page_number}.png"
            pixmap.save(image_path)
            image_paths.append(image_path)

    return image_paths


def preview_page(pdf_path, page_number):
    if not pdf_path or page_number is None:
        return None

    preview_file = NamedTemporaryFile(delete=False, suffix=".png")
    preview_file.close()

    with fitz.open(pdf_path) as pdf:
        page = pdf.load_page(int(page_number) - 1)
        pixmap = page.get_pixmap(matrix=fitz.Matrix(DPI / 72, DPI / 72), alpha=False)
        pixmap.save(preview_file.name)

    return preview_file.name


def load_pdf(pdf_path, start_page):
    if not pdf_path:
        return None, gr.update(value=1)

    with fitz.open(pdf_path) as pdf:
        page_count = pdf.page_count

    return preview_page(pdf_path, start_page), gr.update(value=page_count)


def run_rapidocr(image_paths):
    texts = []

    for image_path in image_paths:
        result = rapidocr(str(image_path))
        texts.append("\n".join(result.txts or []).strip())

    return texts


@lru_cache
def glm_ocr_model():
    from transformers import AutoProcessor, GlmOcrForConditionalGeneration

    model_id = "zai-org/GLM-OCR"
    processor = AutoProcessor.from_pretrained(model_id)
    model = GlmOcrForConditionalGeneration.from_pretrained(model_id, device_map="auto")
    model.eval()
    return processor, model


def run_glm_ocr(image_paths):
    import torch

    processor, model = glm_ocr_model()
    texts = []

    for image_path in image_paths:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "url": str(image_path)},
                    {"type": "text", "text": "Text Recognition:"},
                ],
            }
        ]
        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=2048)

        texts.append(processor.decode(output[0], skip_special_tokens=True).strip())

    return texts


def ocr_pdf(pdf_file, start_page, end_page, engine):
    pages = list(range(int(start_page), int(end_page) + 1))

    with TemporaryDirectory() as image_dir:
        image_paths = render_pages(pdf_file, pages, Path(image_dir))
        if engine == "rapidocr":
            page_texts = run_rapidocr(image_paths)
        else:
            page_texts = run_glm_ocr(image_paths)

    return "\n\n".join(
        f"## Page {page_number}\n\n{text}"
        for page_number, text in zip(pages, page_texts)
    )


with gr.Blocks(title="PDF OCR demo") as demo:
    gr.Markdown("# PDF OCR demo")

    with gr.Row():
        with gr.Column():
            pdf = gr.File(label="PDF", file_types=[".pdf"], type="filepath")

            with gr.Row():
                start_page = gr.Number(label="Start page", value=1, precision=0)
                end_page = gr.Number(label="End page", value=1, precision=0)

            engine = gr.Radio(
                label="OCR engine",
                choices=["rapidocr", "glm_ocr"],
                value="rapidocr",
            )

            button = gr.Button("OCR pages", variant="primary")
            preview = gr.Image(
                label="Page preview",
                type="filepath",
                height=520,
                interactive=False,
            )

        with gr.Column():
            output = gr.Textbox(label="OCR text", lines=24)

    pdf.change(load_pdf, [pdf, start_page], [preview, end_page])
    start_page.change(preview_page, [pdf, start_page], preview)
    button.click(ocr_pdf, [pdf, start_page, end_page, engine], output)


# Does not need a guard


[INFO] 2026-05-24 23:04:34,661 [RapidOCR] base.py:22: Using engine_name: onnxruntime


[INFO] 2026-05-24 23:04:34,720 [RapidOCR] download_file.py:60: File exists and is valid: /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx


[INFO] 2026-05-24 23:04:34,721 [RapidOCR] main.py:57: Using /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.onnx


[INFO] 2026-05-24 23:04:34,779 [RapidOCR] base.py:22: Using engine_name: onnxruntime


[INFO] 2026-05-24 23:04:34,781 [RapidOCR] download_file.py:60: File exists and is valid: /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx


[INFO] 2026-05-24 23:04:34,781 [RapidOCR] main.py:57: Using /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx


[INFO] 2026-05-24 23:04:34,803 [RapidOCR] base.py:22: Using engine_name: onnxruntime


[INFO] 2026-05-24 23:04:34,813 [RapidOCR] download_file.py:60: File exists and is valid: /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx


[INFO] 2026-05-24 23:04:34,813 [RapidOCR] main.py:57: Using /Users/soma/Library/CloudStorage/Dropbox/Soma/Curriculum/2026-dataharvest/ai-experimentation/.venv/lib/python3.11/site-packages/rapidocr/models/ch_PP-OCRv4_rec_mobile.onnx


In [ ]:
demo.launch()
